In [39]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [40]:
import cv2

In [41]:
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dense, Flatten, Dropout

In [42]:
data_path = "archive/Train"


In [43]:
IMG_HEIGHT = 30
IMG_WIDTH = 30
channels = 3

data = []
labels = []
num_classes = 43

In [44]:
for i in range(num_classes):
    path = os.path.join(data_path, str(i))
    images = os.listdir(path)
    
    for img in images:
        try:
            image = cv2.imread(os.path.join(path, img))
            image = cv2.resize(image, (IMG_HEIGHT, IMG_WIDTH))
            data.append(image)
            labels.append(i)
        except:
            print("Error loading image")

data = np.array(data)
labels = np.array(labels)


In [45]:
X_train, X_test, y_train, y_test = train_test_split(
    data, labels, test_size=0.2, random_state=42
)

In [46]:
X_train = X_train / 255.0
X_test = X_test / 255.0

In [47]:
y_train = to_categorical(y_train, num_classes)
y_test = to_categorical(y_test, num_classes)

In [48]:
model = Sequential()

model.add(Conv2D(32, (3,3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, channels)))
model.add(Conv2D(32, (3,3), activation='relu'))
model.add(MaxPool2D(pool_size=(2,2)))
model.add(Dropout(0.25))

model.add(Conv2D(64, (3,3), activation='relu'))
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPool2D(pool_size=(2,2)))
model.add(Dropout(0.25))

model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))

# Compile
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train, y_train,
    batch_size=64,
    epochs=10,
    validation_data=(X_test, y_test)
)

Epoch 1/10
491/491 ━━━━━━━━━━━━━━━━━━━━ 35s 68ms/step - accuracy: 0.5424 - loss: 1.6462 - val_accuracy: 0.9277 - val_loss: 0.2404
Epoch 2/10
491/491 ━━━━━━━━━━━━━━━━━━━━ 33s 67ms/step - accuracy: 0.9098 - loss: 0.2877 - val_accuracy: 0.9802 - val_loss: 0.0843
Epoch 3/10
189/491 ━━━━━━━━━━━━━━━━━━━━ 19s 66ms/step - accuracy: 0.9435 - loss: 0.1909

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title("Model Accuracy")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title("Model Loss")
plt.show()


In [ ]:
model.save("traffic_sign_model.h5")

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("traffic_sign_model.h5")

def predict_image(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (30,30))
    img = np.expand_dims(img, axis=0)
    img = img / 255.0
    
    prediction = model.predict(img)
    class_index = np.argmax(prediction)
    
    print("Predicted Class:", class_index)
    print("100 km/h")

predict_image("test_image3.png")